# Batch Satellite Coverage Analysis

Place this notebook in the same folder that contains `C1`, `C2`, ..., `C15`, then run all cells.

This version is designed for your current file naming convention, for example:

```text
C1/a07562-i40_Satellite_01.csv
C2/a07562-i45_Satellite_01.csv
...
```

It automatically processes all 15 folders, analyzes the four target regions from the table you provided, saves the figures, saves the DOP/coverage data, and creates one combined summary CSV.

The main speed improvement is that the MCI-to-PA conversion is vectorized instead of looping through every time step one at a time. There is also an optional parallel mode.

In [ ]:
# ============================================================
# User settings
# ============================================================

from pathlib import Path

# Put this notebook in the same folder as C1, C2, ..., C15.
# Then Path.cwd() should be that parent folder.
BASE_DIR = Path.cwd()

# Output folder. This will be created automatically.
OUTPUT_ROOT = BASE_DIR / "Coverage_Results"

# The 15 folders to process.
CASE_DIRS = [BASE_DIR / f"C{i}" for i in range(1, 16)]

# Current satellite constellation families to include.
# The notebook will ignore old Tundra/S-ELFO files if they are still in a folder.
CONSTELLATION_PREFIXES = ["a07562", "a12004", "a19056"]

# File matching pattern. This matches names like a07562-i40_Satellite_01.csv
CSV_PATTERN = "*_Satellite_*.csv"

# Visibility mask.
ELEV_MASK_DEG = 5.0

# Lunar radius [km].
R_MOON_KM = 1737.4

# Plot settings.
SAVE_ELEVATION_FIGURES = True
SAVE_VISIBLE_COUNT_FIGURES = True
SAVE_DOP_FIGURES = True
SAVE_SUMMARY_FIGURES = True

# Downsample only for plotting. Computations still use the full data.
MAX_PLOT_POINTS = 3000

# Optional parallel processing.
# Start with False because it is easier to debug.
# After the notebook runs correctly once, you can try True.
USE_PARALLEL = False
MAX_WORKERS = 4

# If True, the notebook will print more information while running.
VERBOSE = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Base directory:", BASE_DIR)
print("Output directory:", OUTPUT_ROOT)

## Target sites from the table

Longitudes are stored with east positive and west negative. Southern latitudes are negative.

In [ ]:
# ============================================================
# Target locations from your table
# ============================================================

TARGET_SITES = {
    # 84°12′05.1″S, 60°1′59.61″E
    "Nobile_Rim_2_DM2": {
        "lat_deg": -(84 + 12/60 + 5.1/3600),
        "lon_deg": 60 + 1/60 + 59.61/3600,
        "label": "Nobile Rim 2 (Site DM2)"
    },

    # ~86.00°S, 2.06°E
    "Malapert_Massif": {
        "lat_deg": -86.00,
        "lon_deg": 2.06,
        "label": "Malapert Massif"
    },

    # 89.1°S, 109.27°W
    "Henson_SSE_Rim": {
        "lat_deg": -89.10,
        "lon_deg": -109.27,
        "label": "Henson SSE Rim"
    },

    # ~87.3°S, 77.0°E
    "Faustini_Rim_A": {
        "lat_deg": -87.30,
        "lon_deg": 77.00,
        "label": "Faustini Rim A"
    },
}

for key, site in TARGET_SITES.items():
    print(f"{key:20s}: lat = {site['lat_deg']:.6f} deg, lon = {site['lon_deg']:.6f} deg")

## Imports and helper utilities

In [ ]:
# ============================================================
# Imports
# ============================================================

import re
import math
import warnings
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    # Fallback if tqdm is unavailable.
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Make sure matplotlib does not leave too many figures open.
plt.rcParams["figure.max_open_warning"] = 0

In [ ]:
# ============================================================
# General helper functions
# ============================================================

def sanitize_name(name):
    """Make a string safe to use in file/folder names."""
    name = str(name).strip()
    name = re.sub(r"[^A-Za-z0-9_\-.]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


def parse_case_name(case_name):
    """
    Parse names like a07562-i40 into useful fields.
    Returns constellation='a07562' and inclination_deg=40.0 when possible.
    """
    match = re.match(r"(?P<constellation>a\d+)-i(?P<inc>[-+]?\d+(?:\.\d+)?)", case_name)
    if match is None:
        return {
            "constellation": case_name,
            "inclination_deg": np.nan
        }

    return {
        "constellation": match.group("constellation"),
        "inclination_deg": float(match.group("inc"))
    }


def get_case_name_from_file(file_path):
    """
    Example:
        a07562-i40_Satellite_01.csv -> a07562-i40
    """
    stem = Path(file_path).stem
    if "_Satellite_" in stem:
        return stem.split("_Satellite_")[0]
    return stem


def group_csv_files_by_case(case_dir, csv_pattern=CSV_PATTERN, constellation_prefixes=None):
    """
    Find all CSV files in a folder and group them by the part before _Satellite_.
    This allows one folder to contain one case or, if needed, multiple cases.
    """
    case_dir = Path(case_dir)
    files = sorted(case_dir.glob(csv_pattern))

    groups = {}
    for file in files:
        case_name = get_case_name_from_file(file)

        if constellation_prefixes is not None:
            keep = any(case_name.startswith(prefix) for prefix in constellation_prefixes)
            if not keep:
                continue

        groups.setdefault(case_name, []).append(file)

    return groups


def find_time_column(df):
    """Handle both the typo EphermerisTime and the standard EphemerisTime."""
    if "EphermerisTime" in df.columns:
        return "EphermerisTime"
    if "EphemerisTime" in df.columns:
        return "EphemerisTime"
    raise KeyError("Could not find EphermerisTime or EphemerisTime column in CSV.")


def get_plot_stride(n_points, max_plot_points=MAX_PLOT_POINTS):
    return max(1, int(np.ceil(n_points / max_plot_points)))

## Vectorized MCI to PA conversion

This is the main speed-up compared with the previous notebook. Instead of calling `mci_to_pa(...)` inside a Python loop for each time step, this version converts all time steps for one satellite at once using NumPy arrays.

In [ ]:
# ============================================================
# Vectorized lunar orientation and MCI -> PA conversion
# ============================================================

def R1_matrix_batch(a):
    """Batch R1 rotation matrices for angle array a, shape (N,)."""
    a = np.asarray(a, dtype=float)
    c = np.cos(a)
    s = np.sin(a)
    R = np.zeros((a.size, 3, 3), dtype=float)
    R[:, 0, 0] = 1.0
    R[:, 1, 1] = c
    R[:, 1, 2] = s
    R[:, 2, 1] = -s
    R[:, 2, 2] = c
    return R


def R3_matrix_batch(a):
    """Batch R3 rotation matrices for angle array a, shape (N,)."""
    a = np.asarray(a, dtype=float)
    c = np.cos(a)
    s = np.sin(a)
    R = np.zeros((a.size, 3, 3), dtype=float)
    R[:, 0, 0] = c
    R[:, 0, 1] = s
    R[:, 1, 0] = -s
    R[:, 1, 1] = c
    R[:, 2, 2] = 1.0
    return R


def R1_matrix_dot_batch(a, a_dot):
    """Batch derivative of R1 rotation matrices."""
    a = np.asarray(a, dtype=float)
    a_dot = np.asarray(a_dot, dtype=float)
    c = np.cos(a)
    s = np.sin(a)
    Rdot = np.zeros((a.size, 3, 3), dtype=float)
    Rdot[:, 1, 1] = -a_dot * s
    Rdot[:, 1, 2] =  a_dot * c
    Rdot[:, 2, 1] = -a_dot * c
    Rdot[:, 2, 2] = -a_dot * s
    return Rdot


def R3_matrix_dot_batch(a, a_dot):
    """Batch derivative of R3 rotation matrices."""
    a = np.asarray(a, dtype=float)
    a_dot = np.asarray(a_dot, dtype=float)
    c = np.cos(a)
    s = np.sin(a)
    Rdot = np.zeros((a.size, 3, 3), dtype=float)
    Rdot[:, 0, 0] = -a_dot * s
    Rdot[:, 0, 1] =  a_dot * c
    Rdot[:, 1, 0] = -a_dot * c
    Rdot[:, 1, 1] = -a_dot * s
    return Rdot


def get_lunar_angles_vectorized(t_tdb):
    """
    Approximate lunar orientation angles based on the same equations used in the previous notebook.

    Parameters
    ----------
    t_tdb : array-like
        Seconds since J2000 epoch.

    Returns
    -------
    phi, theta, psi, phi_dot, theta_dot, psi_dot : arrays
        Euler angles [rad] and rates [rad/s].
    """
    t_tdb = np.asarray(t_tdb, dtype=float)
    t_flat = t_tdb.reshape(-1)

    d = t_flat / 86400.0
    T = d / 36525.0

    E0 = np.array([
        125.045, 250.089, 260.008, 176.625, 357.529,
        311.589, 134.963, 276.617,  34.226,  15.134,
        119.743, 239.961,  25.053
    ], dtype=float)

    Edot = np.array([
        -0.0529921, -0.1059842, 13.0120009, 13.3407154,  0.9856003,
        26.4057084, 13.0649930,  0.3287146,  1.7484877, -0.1589763,
         0.0036096,  0.1643573, 12.9590088
    ], dtype=float)

    E_deg = E0[None, :] + d[:, None] * Edot[None, :]
    E_rad = np.deg2rad(E_deg)
    Edot_rad_per_day = np.deg2rad(Edot)

    a_alpha = np.array([
        -3.8787, -0.1204,  0.0700, -0.0172,  0.0,
         0.0072,  0.0,     0.0,     0.0,    -0.0052,
         0.0,     0.0,     0.0043
    ], dtype=float)

    a_delta = np.array([
         1.5419,  0.0239, -0.0278,  0.0068,  0.0,
        -0.0029,  0.0009,  0.0,     0.0,     0.0008,
         0.0,     0.0,    -0.0009
    ], dtype=float)

    a_W = np.array([
         3.5610,  0.1208, -0.0642,  0.0158,  0.0252,
        -0.0066, -0.0047, -0.0046,  0.0028,  0.0052,
         0.0040,  0.0019, -0.0044
    ], dtype=float)

    alpha0_deg = 269.9949 + 0.0031 * T + np.sum(a_alpha[None, :] * np.sin(E_rad), axis=1)
    delta0_deg =  66.5392 + 0.0130 * T + np.sum(a_delta[None, :] * np.cos(E_rad), axis=1)
    W_deg      =  38.3213 + 13.17635815 * d - 1.4e-12 * d**2 + np.sum(a_W[None, :] * np.sin(E_rad), axis=1)

    dalpha_dd = (
        0.0031 / 36525.0
        + np.sum(a_alpha[None, :] * np.cos(E_rad) * Edot_rad_per_day[None, :], axis=1)
    )

    ddelta_dd = (
        0.0130 / 36525.0
        - np.sum(a_delta[None, :] * np.sin(E_rad) * Edot_rad_per_day[None, :], axis=1)
    )

    dW_dd = (
        13.17635815
        - 2.8e-12 * d
        + np.sum(a_W[None, :] * np.cos(E_rad) * Edot_rad_per_day[None, :], axis=1)
    )

    phi_deg   = 90.0 + alpha0_deg
    theta_deg = 90.0 - delta0_deg
    psi_deg   = W_deg

    dphi_dd   = dalpha_dd
    dtheta_dd = -ddelta_dd
    dpsi_dd   = dW_dd

    phi   = np.deg2rad(phi_deg)
    theta = np.deg2rad(theta_deg)
    psi   = np.deg2rad(psi_deg)

    phi_dot   = np.deg2rad(dphi_dd) / 86400.0
    theta_dot = np.deg2rad(dtheta_dd) / 86400.0
    psi_dot   = np.deg2rad(dpsi_dd) / 86400.0

    return phi, theta, psi, phi_dot, theta_dot, psi_dot


def mci_to_pa_vectorized(t_tdb, x_mci):
    """
    Convert a full time history from MCI to PA.

    Parameters
    ----------
    t_tdb : array, shape (N,)
    x_mci : array, shape (N, 6), columns [x, y, z, vx, vy, vz]

    Returns
    -------
    x_pa : array, shape (N, 6)
    """
    t_tdb = np.asarray(t_tdb, dtype=float)
    x_mci = np.asarray(x_mci, dtype=float)

    if x_mci.ndim != 2 or x_mci.shape[1] != 6:
        raise ValueError("x_mci must have shape (N, 6).")

    phi, theta, psi, phi_dot, theta_dot, psi_dot = get_lunar_angles_vectorized(t_tdb)

    R3_psi = R3_matrix_batch(psi)
    R1_theta = R1_matrix_batch(theta)
    R3_phi = R3_matrix_batch(phi)

    R3_psi_dot = R3_matrix_dot_batch(psi, psi_dot)
    R1_theta_dot = R1_matrix_dot_batch(theta, theta_dot)
    R3_phi_dot = R3_matrix_dot_batch(phi, phi_dot)

    # R_total = R3_psi @ R1_theta @ R3_phi
    R_total = np.einsum("nij,njk,nkl->nil", R3_psi, R1_theta, R3_phi)

    # Product rule derivative.
    R_total_dot = (
        np.einsum("nij,njk,nkl->nil", R3_psi_dot, R1_theta, R3_phi)
        + np.einsum("nij,njk,nkl->nil", R3_psi, R1_theta_dot, R3_phi)
        + np.einsum("nij,njk,nkl->nil", R3_psi, R1_theta, R3_phi_dot)
    )

    r_mci = x_mci[:, :3]
    v_mci = x_mci[:, 3:]

    r_pa = np.einsum("nij,nj->ni", R_total, r_mci)
    v_pa = np.einsum("nij,nj->ni", R_total, v_mci) + np.einsum("nij,nj->ni", R_total_dot, r_mci)

    return np.hstack((r_pa, v_pa))

## Load satellite CSV files and compute visibility/DOP

In [ ]:
# ============================================================
# Loading and coverage functions
# ============================================================

def load_satellite_states_mci(csv_files):
    """Load one case worth of satellite CSV files into a dictionary."""
    sat_states_mci = {}

    for file in sorted(csv_files):
        sat_id = file.stem.split("_")[-1]
        df = pd.read_csv(file)
        time_col = find_time_column(df)

        required_cols = [time_col, "ElapsedTime", "x_MCI", "y_MCI", "z_MCI", "vx_MCI", "vy_MCI", "vz_MCI"]
        missing = [col for col in required_cols if col not in df.columns]
        if missing:
            raise KeyError(f"{file.name} is missing columns: {missing}")

        t_et = df[time_col].to_numpy(dtype=float)
        t_elapsed = df["ElapsedTime"].to_numpy(dtype=float)

        x_mci = df[["x_MCI", "y_MCI", "z_MCI", "vx_MCI", "vy_MCI", "vz_MCI"]].to_numpy(dtype=float)

        sat_states_mci[sat_id] = {
            "source_file": file.name,
            "t_et": t_et,
            "t_elapsed": t_elapsed,
            "x_mci": x_mci,
            "r_mci": x_mci[:, :3],
            "v_mci": x_mci[:, 3:],
        }

    return sat_states_mci


def convert_all_mci_to_pa(sat_states_mci):
    """Convert all satellites in one case from MCI to PA."""
    sat_states_pa = {}

    for sat_id, data in sat_states_mci.items():
        x_pa = mci_to_pa_vectorized(data["t_et"], data["x_mci"])

        sat_states_pa[sat_id] = {
            "source_file": data["source_file"],
            "t_et": data["t_et"],
            "t_elapsed": data["t_elapsed"],
            "x_pa": x_pa,
            "r_pa": x_pa[:, :3],
            "v_pa": x_pa[:, 3:],
        }

    return sat_states_pa


def get_user_pos_pa(lat_deg, lon_deg, R_moon=R_MOON_KM):
    lat = np.deg2rad(lat_deg)
    lon = np.deg2rad(lon_deg)

    return R_moon * np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ])


def get_rot_pa_to_enu(lat_deg, lon_deg):
    lat = np.deg2rad(lat_deg)
    lon = np.deg2rad(lon_deg)

    e_hat = np.array([
        -np.sin(lon),
         np.cos(lon),
         0.0
    ])

    n_hat = np.array([
        -np.sin(lat) * np.cos(lon),
        -np.sin(lat) * np.sin(lon),
         np.cos(lat)
    ])

    u_hat = np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ])

    return np.vstack((e_hat, n_hat, u_hat))


def compute_los_pa(r_sat_pa, r_user_pa):
    rho_pa = r_sat_pa - r_user_pa
    rho_norm = np.linalg.norm(rho_pa, axis=1)
    rho_hat_pa = rho_pa / rho_norm[:, None]
    return rho_pa, rho_hat_pa, rho_norm


def compute_elevation_and_visibility(rho_hat_enu, elev_mask_deg=ELEV_MASK_DEG):
    u_component = rho_hat_enu[:, 2]
    u_component = np.clip(u_component, -1.0, 1.0)

    elevation_rad = np.arcsin(u_component)
    elevation_deg = np.rad2deg(elevation_rad)
    is_visible = elevation_deg > elev_mask_deg

    return elevation_rad, elevation_deg, is_visible


def compute_visibility_for_site(sat_states_pa, lat_deg, lon_deg, elev_mask_deg=ELEV_MASK_DEG):
    """Compute LOS, elevation, and visibility for one surface site."""
    r_user_pa = get_user_pos_pa(lat_deg, lon_deg)
    R_pa_to_enu = get_rot_pa_to_enu(lat_deg, lon_deg)

    sat_los_enu = {}
    sat_visibility = {}

    for sat_id, data in sat_states_pa.items():
        rho_pa, rho_hat_pa, rho_norm = compute_los_pa(data["r_pa"], r_user_pa)
        rho_hat_enu = rho_hat_pa @ R_pa_to_enu.T

        elevation_rad, elevation_deg, is_visible = compute_elevation_and_visibility(
            rho_hat_enu,
            elev_mask_deg=elev_mask_deg
        )

        sat_los_enu[sat_id] = {
            "rho_hat_enu": rho_hat_enu,
            "rho_norm": rho_norm,
        }

        sat_visibility[sat_id] = {
            "elevation_rad": elevation_rad,
            "elevation_deg": elevation_deg,
            "is_visible": is_visible,
        }

    return sat_los_enu, sat_visibility


def stack_case_time_histories(sat_states_pa, sat_los_enu, sat_visibility):
    """
    Stack satellite histories onto a common length.
    This assumes satellites in one case share the same time grid, which is typical for STK exports.
    If one file is shorter, all arrays are truncated to the shortest file length.
    """
    sat_ids = sorted(sat_states_pa.keys())
    N_common = min(len(sat_states_pa[sat_id]["t_elapsed"]) for sat_id in sat_ids)

    # Use first satellite as reference time grid.
    ref_sat = sat_ids[0]
    t_elapsed = sat_states_pa[ref_sat]["t_elapsed"][:N_common]
    t_hours = (t_elapsed - t_elapsed[0]) / 3600.0

    elevation_stack = np.stack(
        [sat_visibility[sat_id]["elevation_deg"][:N_common] for sat_id in sat_ids],
        axis=1
    )

    visibility_stack = np.stack(
        [sat_visibility[sat_id]["is_visible"][:N_common] for sat_id in sat_ids],
        axis=1
    )

    los_enu_stack = np.stack(
        [sat_los_enu[sat_id]["rho_hat_enu"][:N_common, :] for sat_id in sat_ids],
        axis=1
    )

    return sat_ids, t_hours, elevation_stack, visibility_stack, los_enu_stack


def compute_visible_count(visibility_stack):
    return np.sum(visibility_stack, axis=1)


def compute_dop_from_visible_los(rho_hat_enu_visible):
    """Compute DOP values from visible ENU LOS unit vectors."""
    N_visible = rho_hat_enu_visible.shape[0]

    if N_visible < 4:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    G = np.column_stack([
        rho_hat_enu_visible,
        np.ones(N_visible)
    ])

    Q = np.linalg.pinv(G.T @ G)

    q_EE = Q[0, 0]
    q_NN = Q[1, 1]
    q_UU = Q[2, 2]
    q_tt = Q[3, 3]

    GDOP = np.sqrt(q_EE + q_NN + q_UU + q_tt)
    PDOP = np.sqrt(q_EE + q_NN + q_UU)
    HDOP = np.sqrt(q_EE + q_NN)
    VDOP = np.sqrt(q_UU)
    TDOP = np.sqrt(q_tt)

    return GDOP, PDOP, HDOP, VDOP, TDOP


def compute_dop_time_series(visibility_stack, los_enu_stack):
    """Compute DOP arrays over time."""
    N = visibility_stack.shape[0]

    GDOP = np.full(N, np.nan)
    PDOP = np.full(N, np.nan)
    HDOP = np.full(N, np.nan)
    VDOP = np.full(N, np.nan)
    TDOP = np.full(N, np.nan)
    num_visible = np.sum(visibility_stack, axis=1).astype(int)

    for k in range(N):
        if num_visible[k] >= 4:
            visible_los = los_enu_stack[k, visibility_stack[k, :], :]
            GDOP[k], PDOP[k], HDOP[k], VDOP[k], TDOP[k] = compute_dop_from_visible_los(visible_los)

    return GDOP, PDOP, HDOP, VDOP, TDOP, num_visible

## Plotting and saving functions

In [ ]:
# ============================================================
# Plotting functions
# ============================================================

def save_elevation_plot(case_output_dir, full_case_name, site_key, site_label, lat_deg, lon_deg,
                        sat_ids, t_hours, elevation_stack, elev_mask_deg=ELEV_MASK_DEG):
    case_output_dir = Path(case_output_dir)
    sat_groups = [sat_ids[i:i+3] for i in range(0, len(sat_ids), 3)]

    fig, axes = plt.subplots(
        nrows=len(sat_groups),
        ncols=1,
        figsize=(12, max(3.0 * len(sat_groups), 5.0)),
        sharex=True,
        sharey=True
    )

    if len(sat_groups) == 1:
        axes = [axes]

    stride = get_plot_stride(len(t_hours))

    for group_idx, (ax, group) in enumerate(zip(axes, sat_groups)):
        for sat_id in group:
            j = sat_ids.index(sat_id)
            ax.plot(
                t_hours[::stride],
                elevation_stack[::stride, j],
                label=f"Sat {sat_id}",
                linewidth=0.8
            )

        ax.axhline(
            elev_mask_deg,
            linestyle="--",
            color="k",
            linewidth=1.0,
            label=f"{elev_mask_deg:.1f} deg mask"
        )
        ax.set_ylabel("Elevation [deg]")
        ax.set_ylim([-90, 90])
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=4, fontsize=8)

    axes[-1].set_xlabel("Elapsed Time [hours]")

    fig.suptitle(
        f"{full_case_name}: Elevation vs Time\n{site_label}  lat={lat_deg:.4f}°, lon={lon_deg:.4f}°",
        fontsize=13
    )

    plt.tight_layout()
    fig.savefig(case_output_dir / f"{full_case_name}_{site_key}_elevation_vs_time.png", dpi=300)
    plt.close(fig)


def save_visible_count_plot(case_output_dir, full_case_name, site_key, site_label, lat_deg, lon_deg,
                            t_hours, visible_count):
    case_output_dir = Path(case_output_dir)
    stride = get_plot_stride(len(t_hours))

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t_hours[::stride], visible_count[::stride], linewidth=1.2)
    ax.set_xlabel("Elapsed Time [hours]")
    ax.set_ylabel("Number of Visible Satellites")
    ax.set_title(f"{full_case_name}: Coverage Count\n{site_label}  lat={lat_deg:.4f}°, lon={lon_deg:.4f}°")
    ax.grid(True, alpha=0.4)
    ax.set_ylim(bottom=0)

    plt.tight_layout()
    fig.savefig(case_output_dir / f"{full_case_name}_{site_key}_visible_count.png", dpi=300)
    plt.close(fig)


def save_dop_plot(case_output_dir, full_case_name, site_key, site_label, lat_deg, lon_deg,
                  t_hours, GDOP, PDOP, HDOP, VDOP, TDOP):
    case_output_dir = Path(case_output_dir)
    stride = get_plot_stride(len(t_hours))

    # Combined DOP plot.
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(t_hours[::stride], GDOP[::stride], label="GDOP")
    ax.plot(t_hours[::stride], PDOP[::stride], label="PDOP")
    ax.plot(t_hours[::stride], HDOP[::stride], label="HDOP")
    ax.plot(t_hours[::stride], VDOP[::stride], label="VDOP")
    ax.plot(t_hours[::stride], TDOP[::stride], label="TDOP")

    ax.set_xlabel("Elapsed Time [hours]")
    ax.set_ylabel("DOP")
    ax.set_title(f"{full_case_name}: DOP vs Time\n{site_label}  lat={lat_deg:.4f}°, lon={lon_deg:.4f}°")
    ax.grid(True, alpha=0.4)
    ax.legend()

    plt.tight_layout()
    fig.savefig(case_output_dir / f"{full_case_name}_{site_key}_dop_vs_time.png", dpi=300)
    plt.close(fig)

    # 2x2 version like your earlier plot.
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
    dop_data = {
        "GDOP": GDOP,
        "PDOP": PDOP,
        "HDOP": HDOP,
        "VDOP": VDOP,
    }

    for ax, (dop_name, dop_values) in zip(axes.flatten(), dop_data.items()):
        ax.plot(t_hours[::stride], dop_values[::stride], linewidth=1.2)
        ax.set_title(f"{dop_name} Over Time")
        ax.set_ylabel(dop_name)
        ax.grid(True, alpha=0.5)

    axes[1, 0].set_xlabel("Elapsed Time [hours]")
    axes[1, 1].set_xlabel("Elapsed Time [hours]")
    fig.suptitle(f"{full_case_name}: DOP Subplots - {site_label}", fontsize=13)

    plt.tight_layout()
    fig.savefig(case_output_dir / f"{full_case_name}_{site_key}_dop_subplots.png", dpi=300)
    plt.close(fig)


def save_site_timeseries_csv(case_output_dir, full_case_name, site_key, t_hours, visible_count,
                             GDOP, PDOP, HDOP, VDOP, TDOP, num_visible_for_dop):
    case_output_dir = Path(case_output_dir)
    df = pd.DataFrame({
        "t_hours": t_hours,
        "visible_count": visible_count,
        "num_visible_for_dop": num_visible_for_dop,
        "GDOP": GDOP,
        "PDOP": PDOP,
        "HDOP": HDOP,
        "VDOP": VDOP,
        "TDOP": TDOP,
    })
    df.to_csv(case_output_dir / f"{full_case_name}_{site_key}_coverage_dop_timeseries.csv", index=False)
    return df


def summarize_site_results(folder_name, case_name, full_case_name, case_info, site_key, site,
                           num_satellites, t_hours, visible_count, GDOP, PDOP, HDOP, VDOP, TDOP):
    duration_hours = float(t_hours[-1] - t_hours[0]) if len(t_hours) > 1 else 0.0

    def finite_stat(values, func):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            return np.nan
        return float(func(values))

    summary = {
        "folder": folder_name,
        "case_name": case_name,
        "full_case_name": full_case_name,
        "constellation": case_info["constellation"],
        "inclination_deg": case_info["inclination_deg"],
        "target_key": site_key,
        "target_label": site["label"],
        "lat_deg": site["lat_deg"],
        "lon_deg": site["lon_deg"],
        "num_satellites": num_satellites,
        "duration_hours": duration_hours,
        "mean_visible_count": float(np.mean(visible_count)),
        "min_visible_count": float(np.min(visible_count)),
        "max_visible_count": float(np.max(visible_count)),
        "fraction_time_ge_1_visible": float(np.mean(visible_count >= 1)),
        "fraction_time_ge_4_visible": float(np.mean(visible_count >= 4)),
        "mean_GDOP": finite_stat(GDOP, np.mean),
        "median_GDOP": finite_stat(GDOP, np.median),
        "min_GDOP": finite_stat(GDOP, np.min),
        "max_GDOP": finite_stat(GDOP, np.max),
        "mean_PDOP": finite_stat(PDOP, np.mean),
        "median_PDOP": finite_stat(PDOP, np.median),
        "min_PDOP": finite_stat(PDOP, np.min),
        "max_PDOP": finite_stat(PDOP, np.max),
        "mean_HDOP": finite_stat(HDOP, np.mean),
        "mean_VDOP": finite_stat(VDOP, np.mean),
        "mean_TDOP": finite_stat(TDOP, np.mean),
    }

    return summary

## Process one case/folder

For each folder, the notebook:

1. Finds satellite files matching `*_Satellite_*.csv`
2. Groups them by case name, such as `a07562-i40`
3. Converts all 12 satellites from MCI to PA once
4. Reuses those converted satellite states for all four target sites
5. Saves figures, time-series CSV files, and summary rows

In [ ]:
# ============================================================
# Process one case/folder
# ============================================================

def process_one_case(case_dir, case_name, csv_files):
    """Process one constellation/inclination case."""
    case_dir = Path(case_dir)
    folder_name = case_dir.name
    case_info = parse_case_name(case_name)
    full_case_name = sanitize_name(f"{folder_name}_{case_name}")

    case_output_dir = OUTPUT_ROOT / full_case_name
    case_output_dir.mkdir(parents=True, exist_ok=True)

    if VERBOSE:
        print(f"\nProcessing {full_case_name}: {len(csv_files)} satellite files")

    # Load and convert once per case.
    sat_states_mci = load_satellite_states_mci(csv_files)
    sat_states_pa = convert_all_mci_to_pa(sat_states_mci)

    site_summaries = []

    target_iter = TARGET_SITES.items()
    if VERBOSE:
        target_iter = tqdm(list(target_iter), desc=f"Targets for {full_case_name}", leave=False)

    for site_key, site in target_iter:
        site_output_dir = case_output_dir / site_key
        site_output_dir.mkdir(parents=True, exist_ok=True)

        lat_deg = site["lat_deg"]
        lon_deg = site["lon_deg"]
        site_label = site["label"]

        sat_los_enu, sat_visibility = compute_visibility_for_site(
            sat_states_pa,
            lat_deg=lat_deg,
            lon_deg=lon_deg,
            elev_mask_deg=ELEV_MASK_DEG
        )

        sat_ids, t_hours, elevation_stack, visibility_stack, los_enu_stack = stack_case_time_histories(
            sat_states_pa,
            sat_los_enu,
            sat_visibility
        )

        visible_count = compute_visible_count(visibility_stack)
        GDOP, PDOP, HDOP, VDOP, TDOP, num_visible_for_dop = compute_dop_time_series(
            visibility_stack,
            los_enu_stack
        )

        # Save numerical results.
        save_site_timeseries_csv(
            site_output_dir,
            full_case_name,
            site_key,
            t_hours,
            visible_count,
            GDOP,
            PDOP,
            HDOP,
            VDOP,
            TDOP,
            num_visible_for_dop
        )

        # Save figures.
        if SAVE_ELEVATION_FIGURES:
            save_elevation_plot(
                site_output_dir,
                full_case_name,
                site_key,
                site_label,
                lat_deg,
                lon_deg,
                sat_ids,
                t_hours,
                elevation_stack,
                elev_mask_deg=ELEV_MASK_DEG
            )

        if SAVE_VISIBLE_COUNT_FIGURES:
            save_visible_count_plot(
                site_output_dir,
                full_case_name,
                site_key,
                site_label,
                lat_deg,
                lon_deg,
                t_hours,
                visible_count
            )

        if SAVE_DOP_FIGURES:
            save_dop_plot(
                site_output_dir,
                full_case_name,
                site_key,
                site_label,
                lat_deg,
                lon_deg,
                t_hours,
                GDOP,
                PDOP,
                HDOP,
                VDOP,
                TDOP
            )

        summary = summarize_site_results(
            folder_name,
            case_name,
            full_case_name,
            case_info,
            site_key,
            site,
            len(csv_files),
            t_hours,
            visible_count,
            GDOP,
            PDOP,
            HDOP,
            VDOP,
            TDOP
        )
        site_summaries.append(summary)

    return site_summaries


def process_one_folder(case_dir):
    """Process all cases found in one C-folder."""
    case_dir = Path(case_dir)

    if not case_dir.exists():
        return [{
            "folder": case_dir.name,
            "case_name": None,
            "error": f"Folder does not exist: {case_dir}"
        }]

    groups = group_csv_files_by_case(
        case_dir,
        csv_pattern=CSV_PATTERN,
        constellation_prefixes=CONSTELLATION_PREFIXES
    )

    if len(groups) == 0:
        return [{
            "folder": case_dir.name,
            "case_name": None,
            "error": f"No matching satellite CSV files found in {case_dir}"
        }]

    all_summaries = []
    for case_name, csv_files in sorted(groups.items()):
        try:
            case_summaries = process_one_case(case_dir, case_name, csv_files)
            all_summaries.extend(case_summaries)
        except Exception as exc:
            all_summaries.append({
                "folder": case_dir.name,
                "case_name": case_name,
                "error": repr(exc)
            })
            print(f"ERROR in {case_dir.name}/{case_name}: {exc}")

    return all_summaries

## Summary comparison plots

After all folders finish, this cell creates a few quick comparison plots using the combined summary table.

In [ ]:
# ============================================================
# Summary comparison figures
# ============================================================

def save_summary_comparison_plots(summary_df):
    if summary_df.empty:
        return

    if "error" in summary_df.columns:
        good_df = summary_df[summary_df["error"].isna()].copy()
    else:
        good_df = summary_df.copy()

    if good_df.empty:
        print("No successful cases available for summary plots.")
        return

    summary_plot_dir = OUTPUT_ROOT / "Summary_Plots"
    summary_plot_dir.mkdir(parents=True, exist_ok=True)

    # Sort by constellation, inclination, target.
    sort_cols = ["constellation", "inclination_deg", "target_key"]
    good_df = good_df.sort_values(sort_cols)
    good_df["case_label"] = good_df["case_name"]

    metrics = [
        ("mean_visible_count", "Mean Visible Satellite Count"),
        ("fraction_time_ge_1_visible", "Fraction of Time with ≥1 Visible Satellite"),
        ("fraction_time_ge_4_visible", "Fraction of Time with ≥4 Visible Satellites"),
        ("median_PDOP", "Median PDOP"),
    ]

    for metric, ylabel in metrics:
        fig, ax = plt.subplots(figsize=(13, 6))

        for target_key, target_df in good_df.groupby("target_key"):
            target_df = target_df.sort_values(["constellation", "inclination_deg"])
            ax.plot(
                target_df["case_label"],
                target_df[metric],
                marker="o",
                linewidth=1.5,
                label=target_key
            )

        ax.set_xlabel("Case")
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel + " by Case and Target")
        ax.grid(True, alpha=0.35)
        ax.legend(fontsize=8)
        ax.tick_params(axis="x", rotation=45)

        plt.tight_layout()
        fig.savefig(summary_plot_dir / f"summary_{metric}.png", dpi=300)
        plt.close(fig)

    print(f"Saved summary comparison plots to: {summary_plot_dir}")

## Run everything

Run this cell to process all folders. The progress bar updates as folders finish.

Start with `USE_PARALLEL = False`. Once the workflow runs correctly, you can try setting `USE_PARALLEL = True` in the first settings cell.

In [ ]:
# ============================================================
# Run all folders
# ============================================================

all_summaries = []

existing_case_dirs = [case_dir for case_dir in CASE_DIRS if case_dir.exists()]
missing_case_dirs = [case_dir for case_dir in CASE_DIRS if not case_dir.exists()]

if missing_case_dirs:
    print("Missing folders:")
    for case_dir in missing_case_dirs:
        print("  ", case_dir)

if not existing_case_dirs:
    raise FileNotFoundError(
        "No C1-C15 folders were found. Make sure this notebook is located in the folder that contains C1, C2, ..., C15."
    )

print(f"Found {len(existing_case_dirs)} existing C-folders.")

if USE_PARALLEL:
    print(f"Running in parallel with ThreadPoolExecutor, MAX_WORKERS={MAX_WORKERS}")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_one_folder, case_dir): case_dir for case_dir in existing_case_dirs}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Folders complete"):
            case_dir = futures[future]
            try:
                all_summaries.extend(future.result())
            except Exception as exc:
                all_summaries.append({
                    "folder": case_dir.name,
                    "case_name": None,
                    "error": repr(exc)
                })
                print(f"ERROR in folder {case_dir.name}: {exc}")
else:
    print("Running sequentially.")
    for case_dir in tqdm(existing_case_dirs, desc="Folders complete"):
        all_summaries.extend(process_one_folder(case_dir))

summary_df = pd.DataFrame(all_summaries)
summary_path = OUTPUT_ROOT / "coverage_summary_all_cases_targets.csv"
summary_df.to_csv(summary_path, index=False)

print("\nFinished processing.")
print("Saved summary CSV to:", summary_path)

# Show the summary table in the notebook.
display(summary_df)

if SAVE_SUMMARY_FIGURES:
    save_summary_comparison_plots(summary_df)

## Output folder structure

The results will be saved like this:

```text
Coverage_Results/
    C1_a07562-i40/
        Nobile_Rim_2_DM2/
            C1_a07562-i40_Nobile_Rim_2_DM2_elevation_vs_time.png
            C1_a07562-i40_Nobile_Rim_2_DM2_visible_count.png
            C1_a07562-i40_Nobile_Rim_2_DM2_dop_vs_time.png
            C1_a07562-i40_Nobile_Rim_2_DM2_dop_subplots.png
            C1_a07562-i40_Nobile_Rim_2_DM2_coverage_dop_timeseries.csv
        Malapert_Massif/
        Henson_SSE_Rim/
        Faustini_Rim_A/

    C2_a07562-i45/
    ...

    coverage_summary_all_cases_targets.csv
    Summary_Plots/
        summary_mean_visible_count.png
        summary_fraction_time_ge_1_visible.png
        summary_fraction_time_ge_4_visible.png
        summary_median_PDOP.png
```